# 08 · Classic interview patterns (Oracle)

These patterns come up again and again in data engineering interviews:
duplicates, latest row, **gaps and islands** (streaks of consecutive days), retention,
pivot, median, filling missing dates, top N per group.

Most of them use window functions (topic 05). Read the question twice: *ties? NULLs? duplicates?*

**How to run:** Start each SQL cell with `%%oracle`. One statement per cell. Tables are in your own schema, so just write `employees`.

Tables: see [tables_guide.md](../../cheatsheets/tables_guide.md). Run the setup cell first (Shift+Enter).

> **Solutions.** Try the practice notebook first: [`oracle/practice/08_interview_patterns.ipynb`](../../oracle/practice/08_interview_patterns.ipynb)

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "settings.py").exists())   # the repo folder
sys.path.insert(0, str(ROOT / "oracle"))
%load_ext oracle_helper

## Q1. Find the duplicate rows  ·  🟢 easy

Show the rows of `orders` that appear **more than once** (all 5 columns) and `copies` = how many times.

*Expected result: 4 rows · columns: `order_id`, `customer_id`, `order_date`, `amount`, `status`, `copies`*

**How it works:** GROUP BY all columns makes one group per distinct row; `HAVING COUNT(*) > 1` keeps only rows that appear several times.

In [ ]:
%%oracle
SELECT order_id, customer_id, order_date, amount, status, COUNT(*) AS copies
FROM orders
GROUP BY order_id, customer_id, order_date, amount, status
HAVING COUNT(*) > 1

,order_id,customer_id,order_date,amount,status,copies
0,1013,29,2024-01-13,45.0,completed,2
1,1027,18,2024-05-18,170.0,completed,2
2,1095,30,2025-10-14,315.0,returned,2
3,1098,25,2025-11-03,10.0,completed,2


**Why / traps:** GROUP BY **all** columns finds exact duplicates. To find duplicate **keys** (same order_id, different values), group by the key only.

## Q2. Remove duplicates, keep one  ·  🟡 medium

Return `orders` with each exact duplicate kept **once** (104 rows). Use ROW_NUMBER, because that also works when you must choose *which* copy to keep.

*Expected result: 104 rows · columns: `order_id`, `customer_id`, `order_date`, `amount`, `status`*

**How it works:** Number the rows inside each `order_id` with ROW_NUMBER and keep number 1. The ORDER BY decides which copy wins.

In [ ]:
%%oracle
SELECT order_id, customer_id, order_date, amount, status
FROM (SELECT o.*,
             ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY order_date DESC) AS rn
      FROM orders o) t
WHERE rn = 1

,order_id,customer_id,order_date,amount,status
0,1001,27,2023-03-15,135.0,completed
1,1002,7,2023-06-13,45.0,completed
2,1003,3,2023-06-25,70.0,completed
3,1004,19,2023-08-30,10.0,completed
4,1005,27,2023-10-17,180.0,pending
5,1006,30,2023-11-06,155.0,completed
6,1007,18,2023-11-16,94.0,cancelled
7,1008,30,2023-11-23,NULL,completed
8,1009,29,2023-12-23,90.0,completed
9,1010,21,2023-12-25,12.0,completed


**Why / traps:** Simple ways: `SELECT DISTINCT *` or `dropDuplicates()`. ROW_NUMBER lets you keep the **latest** version of a key (e.g. order by `updated_at DESC`).

To delete duplicates in place, Oracle has ROWID (the row's physical address): `DELETE FROM orders WHERE ROWID NOT IN (SELECT MIN(ROWID) FROM orders GROUP BY order_id)`.

## Q3. Login streaks of 3+ days (gaps and islands)  ·  🔴 hard

For each user, find runs of **consecutive** login days that are 3 days or longer. Show `user_id`, `streak_start`, `streak_end`, `days`. A user can log in twice on one day: count the day once.

<details><summary>Hint</summary>

Number each user's days 1, 2, 3... with ROW_NUMBER. `login_date - rn` stays the **same** inside a run of consecutive days. Group by it.

</details>

*Expected result: 31 rows · columns: `user_id`, `streak_start`, `streak_end`, `days`*

**How it works:** 1) One row per user per day (DISTINCT). 2) ROW_NUMBER per user by date. 3) `date − row_number` is the same for consecutive days = a group id. 4) Group by it: MIN, MAX, COUNT.

In [ ]:
%%oracle
WITH days AS (
    SELECT DISTINCT user_id, login_date FROM user_logins
),
numbered AS (
    SELECT user_id, login_date,
           ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY login_date) AS rn
    FROM days
),
grouped AS (
    SELECT user_id, login_date, login_date - rn AS grp
    FROM numbered
)
SELECT user_id, MIN(login_date) AS streak_start, MAX(login_date) AS streak_end, COUNT(*) AS days
FROM grouped
GROUP BY user_id, grp
HAVING COUNT(*) >= 3
ORDER BY user_id, streak_start

,user_id,streak_start,streak_end,days
0,1,2025-01-12,2025-01-18,7
1,1,2025-02-13,2025-02-16,4
2,1,2025-02-23,2025-02-26,4
3,2,2025-01-07,2025-01-13,7
4,2,2025-01-18,2025-01-24,7
5,2,2025-02-03,2025-02-05,3
6,2,2025-02-12,2025-02-18,7
7,2,2025-02-22,2025-02-25,4
8,3,2025-01-12,2025-01-21,10
9,3,2025-02-03,2025-02-07,5


**Why / traps:** **Gaps and islands**: an island = a run of consecutive values. Date minus row number is constant inside an island. Remove same-day duplicates first, or the trick breaks.

## Q4. Longest streak per user  ·  🔴 hard

Show `user_id` and `longest_streak` (days) for every user who logged in.

*Expected result: 8 rows · columns: `user_id`, `longest_streak`*

**How it works:** Same group trick as Q3; count days per group, then take the MAX per user.

In [ ]:
%%oracle
WITH days AS (SELECT DISTINCT user_id, login_date FROM user_logins),
grouped AS (
    SELECT user_id,
           login_date - ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY login_date) AS grp
    FROM days
),
streaks AS (SELECT user_id, COUNT(*) AS days FROM grouped GROUP BY user_id, grp)
SELECT user_id, MAX(days) AS longest_streak
FROM streaks
GROUP BY user_id
ORDER BY user_id

,user_id,longest_streak
0,1,7
1,2,7
2,3,10
3,4,14
4,5,7
5,6,10
6,7,10
7,8,10


## Q5. Find the gaps  ·  🟡 medium

For each user, show the gaps between logins: `user_id`, `last_login` (day before the gap), `next_login` (day after the gap), `days_missed`. Only gaps where at least 1 day was missed.

*Expected result: 46 rows · columns: `user_id`, `last_login`, `next_login`, `days_missed`*

**How it works:** `LEAD(login_date)` gives the user's next login day; when next − this > 1, days were missed in between.

In [ ]:
%%oracle
WITH days AS (SELECT DISTINCT user_id, login_date FROM user_logins),
nxt AS (
    SELECT user_id, login_date AS last_login,
           LEAD(login_date) OVER (PARTITION BY user_id ORDER BY login_date) AS next_login
    FROM days
)
SELECT user_id, last_login, next_login, next_login - last_login - 1 AS days_missed
FROM nxt
WHERE next_login - last_login > 1
ORDER BY user_id, last_login

,user_id,last_login,next_login,days_missed
0,1,2025-01-07,2025-01-12,4
1,1,2025-01-18,2025-01-21,2
2,1,2025-01-22,2025-01-26,3
3,1,2025-01-27,2025-02-03,6
4,1,2025-02-03,2025-02-07,3
5,1,2025-02-07,2025-02-13,5
6,1,2025-02-16,2025-02-23,6
7,2,2025-01-02,2025-01-07,4
8,2,2025-01-13,2025-01-18,4
9,2,2025-01-24,2025-01-30,5


**Why / traps:** The last login of each user has `next_login = NULL`, so the WHERE drops it (NULL > 1 is unknown).

## Q6. Retention: January users back in February  ·  🟡 medium

Of the users who logged in during **January 2025**, how many also logged in during **February 2025**? Show `jan_users`, `returned_users`, `retention_pct` (1 decimal).

*Expected result: 1 row · columns: `jan_users`, `returned_users`, `retention_pct`*

**How it works:** Users of January, users of February. LEFT JOIN January → February: `COUNT(feb.user_id)` counts only those who came back.

In [ ]:
%%oracle
WITH jan AS (
    SELECT DISTINCT user_id FROM user_logins
    WHERE login_date >= DATE '2025-01-01' AND login_date < DATE '2025-02-01'
),
feb AS (
    SELECT DISTINCT user_id FROM user_logins
    WHERE login_date >= DATE '2025-02-01' AND login_date < DATE '2025-03-01'
)
SELECT COUNT(*)                                       AS jan_users,
       COUNT(feb.user_id)                             AS returned_users,
       ROUND(100 * COUNT(feb.user_id) / COUNT(*), 1)  AS retention_pct
FROM jan
LEFT JOIN feb ON feb.user_id = jan.user_id

,jan_users,returned_users,retention_pct
0,8,8,100


**Why / traps:** LEFT JOIN from the **starting group**, then `COUNT(right_key)` counts only the matches.

## Q7. Pivot: items sold per category and year  ·  🟡 medium

Show one row per `category` with columns `y2023`, `y2024`, `y2025` = quantity sold in **completed** orders (each order once). Use 0 when nothing was sold.

*Expected result: 4 rows · columns: `category`, `y2023`, `y2024`, `y2025`*

**How it works:** One SUM(CASE ...) per year turns year **rows** into year **columns**; ELSE 0 gives 0 for missing years.

In [ ]:
%%oracle
WITH
sold AS (
    SELECT p.category, p.product_name, EXTRACT(YEAR FROM o.order_date) AS yr, i.quantity
    FROM (SELECT DISTINCT order_id, order_date, status FROM orders) o
    JOIN order_items i ON i.order_id = o.order_id
    JOIN products p    ON p.product_id = i.product_id
    WHERE o.status = 'completed'
)
SELECT category,
       SUM(CASE WHEN yr = 2023 THEN quantity ELSE 0 END) AS y2023,
       SUM(CASE WHEN yr = 2024 THEN quantity ELSE 0 END) AS y2024,
       SUM(CASE WHEN yr = 2025 THEN quantity ELSE 0 END) AS y2025
FROM sold
GROUP BY category
ORDER BY category

,category,y2023,y2024,y2025
0,Books,7,25,23
1,Clothing,5,14,26
2,Electronics,2,23,27
3,Home,5,23,20


**Why / traps:** CASE inside SUM is the portable pivot. All 3 engines also have a `PIVOT` clause, but its syntax is different in each one.

Oracle: `... PIVOT (SUM(quantity) FOR yr IN (2023 AS y2023, 2024 AS y2024))`.

## Q8. Median salary per department  ·  🟡 medium

Show `dept_id` and `median_salary` for each department (skip NULL dept_id).

*Expected result: 6 rows · columns: `dept_id`, `median_salary`*

**How it works:** Median = 50th percentile: `MEDIAN()` (Oracle, Spark) or `PERCENTILE_CONT(salary, 0.5)` window (BigQuery).

In [ ]:
%%oracle
SELECT dept_id, MEDIAN(salary) AS median_salary
FROM employees
WHERE dept_id IS NOT NULL
GROUP BY dept_id
ORDER BY dept_id

,dept_id,median_salary
0,1,120000
1,2,90000
2,3,85000
3,4,99000
4,5,72000
5,6,75000


**Why / traps:** Median = the middle value; with an even count, the average of the two middle values. Without a built-in: ROW_NUMBER + COUNT, then average the middle row(s).

Also: `PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY salary)`.

## Q9. Fill missing dates with 0  ·  🔴 hard

For **user 1**, show every date of January 2025 (`login_date`) and `num_logins` (login rows that day). Days without logins show **0**. 31 rows.

*Expected result: 31 rows · columns: `login_date`, `num_logins`*

**How it works:** 1) Generate all 31 dates. 2) LEFT JOIN the logins of user 1 **in the ON clause**. 3) COUNT the matches per date (0 when none).

In [ ]:
%%oracle
WITH cal AS (
    SELECT DATE '2025-01-01' + LEVEL - 1 AS d
    FROM dual
    CONNECT BY LEVEL <= 31
)
SELECT cal.d AS login_date, COUNT(l.user_id) AS num_logins
FROM cal
LEFT JOIN user_logins l
       ON l.login_date = cal.d AND l.user_id = 1
GROUP BY cal.d
ORDER BY cal.d

,login_date,num_logins
0,2025-01-01,0
1,2025-01-02,0
2,2025-01-03,0
3,2025-01-04,0
4,2025-01-05,0
5,2025-01-06,2
6,2025-01-07,1
7,2025-01-08,0
8,2025-01-09,0
9,2025-01-10,0


**Why / traps:** Put the `user_id = 1` filter in the **ON** clause. In WHERE it would remove the empty days (their user_id is NULL), and the LEFT JOIN becomes an INNER JOIN.

`CONNECT BY LEVEL <= n` makes n rows from `dual`: the classic Oracle row generator.

## Q10. New users per day and running total  ·  🟡 medium

A user is **new** on the day of their first login. Show `login_date`, `new_users`, `total_users` (running total), only for days with at least one new user.

*Expected result: 4 rows · columns: `login_date`, `new_users`, `total_users`*

**How it works:** 1) First login day per user (MIN). 2) Count users per first day. 3) Running SUM of those counts.

In [ ]:
%%oracle
WITH firsts AS (
    SELECT user_id, MIN(login_date) AS first_day
    FROM user_logins
    GROUP BY user_id
)
SELECT first_day AS login_date,
       COUNT(*)  AS new_users,
       SUM(COUNT(*)) OVER (ORDER BY first_day) AS total_users
FROM firsts
GROUP BY first_day
ORDER BY first_day

,login_date,new_users,total_users
0,2025-01-01,1,1
1,2025-01-03,2,3
2,2025-01-05,1,4
3,2025-01-06,4,8


**Why / traps:** `SUM(COUNT(*)) OVER (...)`: the window runs **after** GROUP BY, so it can use an aggregate.

## Q11. Best-selling product per category (with ties)  ·  🟡 medium

For each category, show the product(s) with the **most quantity sold** in completed orders (each order once). Show `category`, `product_name`, `qty`. Keep ties.

*Expected result: 5 rows · columns: `category`, `product_name`, `qty`*

**How it works:** 1) Quantity per product (completed, unique orders). 2) RANK per category by quantity. 3) Keep rank 1 (ties stay).

In [ ]:
%%oracle
WITH
sold AS (
    SELECT p.category, p.product_name, EXTRACT(YEAR FROM o.order_date) AS yr, i.quantity
    FROM (SELECT DISTINCT order_id, order_date, status FROM orders) o
    JOIN order_items i ON i.order_id = o.order_id
    JOIN products p    ON p.product_id = i.product_id
    WHERE o.status = 'completed'
),
per_product AS (
    SELECT category, product_name, SUM(quantity) AS qty
    FROM sold
    GROUP BY category, product_name
),
ranked AS (
    SELECT category, product_name, qty,
           RANK() OVER (PARTITION BY category ORDER BY qty DESC) AS rnk
    FROM per_product
)
SELECT category, product_name, qty
FROM ranked
WHERE rnk = 1
ORDER BY category, product_name

,category,product_name,qty
0,Books,Data Engineering 101,17
1,Clothing,Socks Pack,18
2,Electronics,Laptop Stand,12
3,Home,Coffee Mug,16
4,Home,Water Bottle,16


**Why / traps:** Aggregate first, rank second. RANK keeps ties.

## Q12. Two orders in a row with the same status  ·  🟡 medium

For each customer, sort their different orders by date (then order_id). Find orders whose status is the **same as the previous order's** status. Show `customer_id`, `order_id`, `order_date`, `status`.

*Expected result: 35 rows · columns: `customer_id`, `order_id`, `order_date`, `status`*

**How it works:** `LAG(status)` over each customer's orders gives the previous status; keep rows where status = previous status.

In [ ]:
%%oracle
WITH o AS (
    SELECT DISTINCT order_id, customer_id, order_date, status FROM orders
),
with_prev AS (
    SELECT o.*,
           LAG(status) OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS prev_status
    FROM o
)
SELECT customer_id, order_id, order_date, status
FROM with_prev
WHERE status = prev_status
ORDER BY customer_id, order_date, order_id

,customer_id,order_id,order_date,status
0,9,1071,2025-06-16,completed
1,10,1093,2025-09-28,completed
2,12,1063,2025-04-18,completed
3,14,1079,2025-07-15,completed
4,17,1083,2025-07-31,completed
5,18,1027,2024-05-18,completed
6,20,1021,2024-03-22,completed
7,20,1024,2024-04-10,completed
8,21,1012,2024-01-08,completed
9,22,1082,2025-07-29,completed
